# Generation Stage — Notebook A (0%–25%)

Generates LLM responses for each query-method combination using GPT-4o-mini. For each run, the retrieved documents are passed to the LLM in retrieval order as numbered search results, and the LLM generates a product recommendation with inline citations. Since the generation is time-consuming, the workload is split across four parallel notebooks (A–D), each processing its own share of all runs.

**Input:**
- `data/retail/retrieval/20260413_retrieval_results_geo_v2.csv` — retrieval results per query-method combination

**Output:**
- `data/retail/generation/4_generation_results_A_v4.parquet` — generation results for this notebook's range
- `data/retail/generation/4_generation_results_v4.parquet` — merged result after all four notebooks complete

## Parallel Setup
| Notebook | Range |
|---|---|
| A | 0% – 25% |
| B | 25% – 50% |
| C | 50% – 75% |
| D | 75% – 100% |

## Structure
1. **Setup** — imports and paths
2. **Data Preparation** — loads retrieval CSV and identifies runs with/without target
3. **Helper Functions** — prompt builder, citation extractor
4. **Parameters** — LLM model, output path, run assignment
5. **Generation** — runs all assigned query-method combinations, saves incrementally
6. **Inspect Results** — verifies output for a sample run
7. **Merge Results** — combines A–D into one parquet (run after all four complete)
8. **Summary** — verifies completeness of merged output

## 1. Setup

Imports required libraries and defines file paths relative to the project root.

In [1]:
# Library imports
import json
import os
import re
import sys
import time
from datetime import datetime

import pandas as pd
import numpy as np

# Go up two levels to project root
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, "..", ".."))

# Add src/ to path so OpenAIHelper can be imported
sys.path.insert(0, os.path.join(project_root, "src"))


from llms import OpenAIHelper

# Load API key from config.json
config_path = os.path.join(project_root, "config.json")
with open(config_path, "r") as f:
    config = json.load(f)
os.environ["OPENAI_API_KEY"] = config["OPENAI_API_KEY"]

data_dir = os.path.join(project_root, "data", "retail")

print("Setup complete.")
print(f"Project root: {project_root}")

Setup complete.
Project root: /Users/leonardrampf/Library/CloudStorage/OneDrive-Personal/Dokumente/Universität/Nova SBE/Work Project/geo-experiment


## 2. Data Preparation

Loads the retrieval CSV and parses the compound `query_id` (format: `{original_query_id}_{method}`) into its components. Identifies which runs have a target document in the retrieval results and which do not — runs without a target are skipped during generation.

In [2]:
# Load the retrieval CSV — one row per document per query_id
# query_id format: "{original_query_id}_{method}"
retrieval_csv_path = os.path.join(data_dir, "retrieval/20260413_retrieval_results_geo_v2.csv")
df = pd.read_csv(retrieval_csv_path)

# Extract original_query_id (numeric) and method name from compound query_id
df["original_query_id"] = df["query_id"].str.extract(r"^(\d+)_")
df["method"]            = df["query_id"].str.extract(r"^\d+_(.+)$")

# Identify runs where the target document is missing from the retrieval results
# These runs are skipped during generation (no valid citation position can be computed)
target_per_run = df.groupby("query_id")["is_target"].sum()
runs_with_target    = set(target_per_run[target_per_run == 1].index)
runs_without_target = set(target_per_run[target_per_run == 0].index)

# Save runs without target for documentation and reproducibility
runs_without_target_path = os.path.join(data_dir, "generation/4_generation_runs_without_target_v3.json")
with open(runs_without_target_path, "w") as f:
    json.dump(sorted(list(runs_without_target)), f, indent=4)

print(f"Total rows:              {len(df)}")
print(f"Unique runs (query_ids): {df['query_id'].nunique()}")
print(f"Unique original queries: {df['original_query_id'].nunique()}")
print(f"Unique methods:          {df['method'].nunique()}")
print(f"Runs with target:        {len(runs_with_target)}")
print(f"Runs without target:     {len(runs_without_target)} — saved to generation/4_generation_runs_without_target_v3.json")
print()
print("Methods found:")
for m in sorted(df['method'].unique()):
    print(f"  {m}")

Total rows:              109875
Unique runs (query_ids): 11000
Unique original queries: 500
Unique methods:          22
Runs with target:        10985
Runs without target:     15 — saved to generation/4_generation_runs_without_target_v3.json

Methods found:
  Authoritative(doc)
  CQ(doc)
  CQS(doc)
  CS(doc)
  Citations(doc)
  ContentImprovement(doc)
  FC(doc)
  FCQ(doc)
  FCQS(doc)
  FCS(doc)
  FQ(doc)
  FQS(doc)
  FS(doc)
  Fluency(doc)
  LLMstxt(doc)
  QS(doc)
  Quotes(doc)
  SimpleLanguage(doc)
  Statistics(doc)
  TechnicalTerms(doc)
  UniqueWords(doc)
  doc


## 3. Helper Functions

Defines the developer prompt and three helper functions used during generation:
- `build_context` — sorts retrieved documents by rank and identifies the 0-based target position
- `build_user_prompt` — formats the query and numbered document list into the LLM user prompt
- `extract_citation_order` — parses inline citations from the LLM response and returns them as a 0-based index list in order of first appearance

In [3]:
# System prompt for the LLM — instructs it to act as a product recommender, recommend multiple products, and cite each product with exactly one inline reference [1].
DEVELOPER_PROMPT = """Your are a product recommender system for an online marketplace. Write an accurate and concise answer for the given user question, using only the provided summarized web search results. The answer should be correct, high-quality, and written by an expert using an unbiased and journalistic tone. The answer should be informative, interesting, and engaging. The answer's logic and reasoning should be rigorous and defensible. Each search result represents exactly one product. The cited search result should fully support all the information about that product. Search results need to be cited using [index], for example [1]. Recommend multiple products, each based on exactly one search result. Do not stack multiple citations for a single product."""

# Document type label used in the user prompt
DOC_TYPE = "Product Description"


def build_context(group):
    """
    Takes all rows for one query_id, sorts by rank, re-numbers 1..N.
    Returns:
        docs: list of doc_text in retrieval order
        target_new_position: 0-based position of target doc after re-numbering
    """
    group = group.sort_values("rank").reset_index(drop=True)
    docs = group["doc_text"].tolist()

    # Find 0-based position of the target document
    target_new_position = None
    for i, row in group.iterrows():
        if row["is_target"] == 1:
            target_new_position = i  # 0-based
            break

    return docs, target_new_position


def build_user_prompt(query, docs):
    """
    Builds the user prompt for the LLM.
    docs: list of doc_text strings, numbered 1..N
    """
    sources = ""
    for i, doc_text in enumerate(docs, start=1):
        sources += f"[{i}] {DOC_TYPE}: {doc_text}\n\n"
    return f"Question: {query}\n\nSearch Results:\n{sources}"


def extract_citation_order(response_text):
    """
    Extracts citation indices in order of first appearance.
    0-based: [1] in text -> index 0, [2] -> index 1, etc.
    e.g. "...product [2] is great [1][2]..." -> [1, 0]
    """
    citations = re.findall(r"\[(\d+)\]", response_text)
    seen = []
    for c in citations:
        idx = int(c) - 1  # 0-based
        if idx not in seen:
            seen.append(idx)
    return seen


print("Helper functions defined.")

Helper functions defined.


## 4. Parameters

Sets the LLM model, output path, and LLM inference parameters. Automatically assigns this notebook's share of all runs (0%–25%), loads already completed runs from the output parquet to support safe resume after interruption, and initialises the LLM client.

In [4]:
# LLM model used for response generation
LLM_NAME = "gpt-4o-mini-2024-07-18"

# Notebook ID — A processes 0%–25% of all runs
NOTEBOOK_ID = "A"

# Output file for this notebook's results — separate from B, C, D to allow parallel execution
output_path = os.path.join(data_dir, "generation/4_generation_results_A_v4.parquet")

# Based on Vertex AI / Gemini default inference parameters:
# https://docs.cloud.google.com/vertex-ai/generative-ai/docs/model-reference/inference?hl=de
TEMPERATURE       = 1.0
TOP_P             = 0.95
SEED              = 42
FREQUENCY_PENALTY = 0.0
PRESENCE_PENALTY  = 0.0

# Assign this notebook's quarter of all runs sorted alphabetically by query_id
all_run_ids_full = sorted(runs_with_target)
total = len(all_run_ids_full)
start_idx = int(total * 0/100)
end_idx   = int(total * 25/100)
all_run_ids = all_run_ids_full[start_idx:end_idx]

# Load already completed runs to support safe resume after interruption
if os.path.exists(output_path):
    df_done = pd.read_parquet(output_path)
    done_keys = set(df_done["query_id"].tolist())
    already_done = len(done_keys)
else:
    done_keys = set()
    already_done = 0

llm = OpenAIHelper(LLM_NAME)

remaining = len(all_run_ids) - already_done
print(f"Notebook:      {NOTEBOOK_ID}")
print(f"LLM:           {LLM_NAME}")
print(f"Range:         0%–25% ({start_idx}–{end_idx} of {total})")
print(f"Runs in range: {len(all_run_ids)}")
print(f"Already done:  {already_done}")
print(f"Remaining:     {remaining}")


Notebook:      A
LLM:           gpt-4o-mini-2024-07-18
Range:         0%–25% (0–2746 of 10985)
Runs in range: 2746
Already done:  0
Remaining:     2746


## 5. Generation

Iterates over all assigned runs and generates one LLM response per query-method combination. For each run, the retrieved documents are sorted by rank, passed to the LLM as numbered search results, and the response is parsed for inline citations. Results are saved incrementally after each run — safe to interrupt and resume. Runs without a target document in the retrieval results are skipped. Failed runs are retried once after a 10-second wait.

In [6]:
# Load existing results if the output file already exists
# Allows safe resume without reprocessing completed runs
if os.path.exists(output_path):
    results_df = pd.read_parquet(output_path)
    results = results_df.to_dict("records")
else:
    results = []

start_time = datetime.now()
print(f"Started at: {start_time.strftime('%H:%M:%S')}")
print(f"Total runs to process: {len(all_run_ids)}")
print()

for i, query_id in enumerate(all_run_ids):

    # Skip runs already completed in a previous session
    if query_id in done_keys:
        continue

    # Load all documents for this run
    group = df[df["query_id"] == query_id]
    query_text         = group["query"].iloc[0]
    original_query_id  = group["original_query_id"].iloc[0]
    method             = group["method"].iloc[0]

    # Sort docs by retrieval rank and find target position
    docs, target_new_position = build_context(group)

    # Skip runs where the target document is not in the retrieval results
    if target_new_position is None:
        print(f"  [{i+1}] {query_id}: no target — skipping")
        continue

    # Build LLM prompt with numbered search results
    user_prompt = build_user_prompt(query_text, docs)
    messages = [
        {"role": "system", "content": DEVELOPER_PROMPT},
        {"role": "user",   "content": user_prompt},
    ]

    try:
        # Generate LLM response and extract citation order
        response, _ = llm.generate(messages)
        response_text    = response.content
        citation_order   = extract_citation_order(response_text)

        # Store result and save incrementally — no work lost on crash or interruption
        results.append({
            "query_id":            query_id,
            "original_query_id":   original_query_id,
            "method":              method,
            "query":               query_text,
            "target_new_position": target_new_position,
            "num_docs":            len(docs),
            "citation_order":      citation_order,
            "llm_response":        response_text,
        })
        done_keys.add(query_id)
        pd.DataFrame(results).to_parquet(output_path, index=False)
        print(f"  [{i+1}/{len(all_run_ids)}] {query_id}: target at [{target_new_position}] | cited: {citation_order[:5]}")

    except Exception as e:
        # First failure — wait 10 seconds and retry once (handles rate limits)
        print(f"  [{i+1}/{len(all_run_ids)}] {query_id}: ERROR — {e}")
        time.sleep(10)
        try:
            response, _    = llm.generate(messages)
            response_text  = response.content
            citation_order = extract_citation_order(response_text)
            results.append({
                "query_id":            query_id,
                "original_query_id":   original_query_id,
                "method":              method,
                "query":               query_text,
                "target_new_position": target_new_position,
                "num_docs":            len(docs),
                "citation_order":      citation_order,
                "llm_response":        response_text,
            })
            done_keys.add(query_id)
            pd.DataFrame(results).to_parquet(output_path, index=False)
            print(f"  [{i+1}/{len(all_run_ids)}] {query_id}: done (retry OK)")
        except Exception as e2:
            # Second failure — skip this run and continue with the next
            print(f"  [{i+1}/{len(all_run_ids)}] {query_id}: FAILED — {e2}")

# Summary
end_time = datetime.now()
elapsed  = end_time - start_time
print(f"{'='*60}")
print(f"GENERATION COMPLETE")
print(f"Started:    {start_time.strftime('%H:%M:%S')}")
print(f"Finished:   {end_time.strftime('%H:%M:%S')}")
print(f"Total time: {str(elapsed).split('.')[0]}")
print(f"Results:    {len(results)} rows saved to {output_path}")

Started at: 21:43:13
Total runs to process: 2746

  [1268/2746] 110544_Fluency(doc): target at [0] | cited: [0, 1, 2, 8, 6]
  [1269/2746] 110544_LLMstxt(doc): target at [8] | cited: [0, 1, 6, 4, 5]
  [1270/2746] 110544_QS(doc): target at [5] | cited: [0, 1, 4, 3, 8]
  [1271/2746] 110544_Quotes(doc): target at [7] | cited: [0, 1, 2, 7, 6]
  [1272/2746] 110544_SimpleLanguage(doc): target at [0] | cited: [0, 1, 2, 5, 6]
  [1273/2746] 110544_Statistics(doc): target at [7] | cited: [0, 1, 2, 4, 7]
  [1274/2746] 110544_TechnicalTerms(doc): target at [6] | cited: [0, 1, 2, 6, 7]
  [1275/2746] 110544_UniqueWords(doc): target at [6] | cited: [0, 1, 3, 6, 7]
  [1276/2746] 110544_doc: target at [1] | cited: [0, 1, 2, 5, 6]
  [1277/2746] 110546_Authoritative(doc): target at [5] | cited: [0, 1, 2, 8, 9]
  [1278/2746] 110546_CQ(doc): target at [5] | cited: [0, 1, 2, 3, 4]
  [1279/2746] 110546_CQS(doc): target at [5] | cited: [0, 1, 2, 3, 4]
  [1280/2746] 110546_CS(doc): target at [5] | cited: [0, 1,

## 6. Inspect Results

Loads this notebook's parquet file and prints a summary for a single run. Change `INSPECT_QUERY_ID` to inspect any specific run.

In [7]:
# Load this notebook's parquet file for inspection
df_results = pd.read_parquet(output_path)

# Change INSPECT_QUERY_ID to inspect a specific run
INSPECT_QUERY_ID = df_results["query_id"].iloc[0]

row = df_results[df_results["query_id"] == INSPECT_QUERY_ID].iloc[0]

print(f"Run (query_id):    {row['query_id']}")
print(f"Original query:    {row['original_query_id']}")
print(f"Method:            {row['method']}")
print(f"Query:             {row['query']}")
print(f"Num docs:          {row['num_docs']}")
print(f"Target position:   {row['target_new_position']} (0-based)")
print(f"Citation order:    {row['citation_order']}")
print(f"Target cited:      {'YES' if row['target_new_position'] in row['citation_order'] else 'NO'}")
print(f"\n--- LLM Response ---")
print(row["llm_response"][:800])

Run (query_id):    100162_Authoritative(doc)
Original query:    100162
Method:            Authoritative(doc)
Query:             table lamps for bedroom
Num docs:          10
Target position:   7 (0-based)
Citation order:    [0 1 3 4 6 9]
Target cited:      NO

--- LLM Response ---
When selecting table lamps for your bedroom, several excellent options are available that combine functionality with stylish design. Here are some noteworthy recommendations:

1. **Touch Control Table Lamp with USB Ports**: This modern lamp features a 3-way dimmable touch control, allowing you to adjust brightness levels easily. It includes two USB charging ports, enabling you to charge your devices even when the lamp is off. The lamp is designed with a sleek white linen shade and a high-quality black metal base, making it an aesthetically pleasing addition to any room [1].

2. **HAITRAL Bedside Table Lamps Set of 2**: This set features two compact lamps with a classic modern design, constructed from a metal 

## 7. Merge Results

Combines the four notebook parquets (A–D) into a single merged file `4_generation_results_v4.parquet`. Run this only after all four notebooks have completed.

In [8]:
# Merge all four notebook parquet files into one combined file
# Run only after notebooks A, B, C, D have all completed
NOTEBOOK_IDS = ["A", "B", "C", "D"]
dfs = []
for nb in NOTEBOOK_IDS:
    path = os.path.join(data_dir, f"generation/4_generation_results_{nb}_v4.parquet")
    if os.path.exists(path):
        dfs.append(pd.read_parquet(path))
        print(f"Loaded 4_generation_results_{nb}_v4.parquet ({len(dfs[-1])} rows)")
    else:
        print(f"WARNING: 4_generation_results_{nb}_v4.parquet not found — skipping")

# Concatenate all four subsets into one merged DataFrame
merged      = pd.concat(dfs, ignore_index=True)
merged_path = os.path.join(data_dir, "generation/4_generation_results_v4.parquet")
merged.to_parquet(merged_path, index=False)
print(f"\nMerged {len(merged)} rows into 4_generation_results_v4.parquet")

Loaded 4_generation_results_A_v4.parquet (2746 rows)
Loaded 4_generation_results_B_v4.parquet (2746 rows)
Loaded 4_generation_results_C_v4.parquet (2746 rows)
Loaded 4_generation_results_D_v4.parquet (2747 rows)

Merged 10985 rows into 4_generation_results_v4.parquet


## 8. Summary

Loads the merged parquet and verifies completeness — checks total runs, unique methods, and unique queries against the expected counts.

In [9]:
# Load merged parquet and verify completeness
# Expected: total rows == total runs with target across all 4 notebooks
merged_path = os.path.join(data_dir, "generation/4_generation_results_v4.parquet")
df_results  = pd.read_parquet(merged_path)

print(f"Total runs done:     {len(df_results)}")
print(f"Total runs expected: {len(all_run_ids_full)}")
print(f"Missing runs:        {len(all_run_ids_full) - len(df_results)}")
print()
print(f"Unique methods: {df_results['method'].nunique()}")
print(f"Unique queries: {df_results['original_query_id'].nunique()}")

Total runs done:     10985
Total runs expected: 10985
Missing runs:        0

Unique methods: 22
Unique queries: 500


In [10]:
df_sample = pd.read_parquet(merged_path)

examples = df_sample[df_sample["method"] == "doc"].sample(5, random_state=42).iloc[1:]

for _, row in examples.iterrows():
    print("=" * 80)
    print(f"Query ID: {row['original_query_id']}")
    print(f"Query:    {row['query']}")
    print(f"Num docs: {row['num_docs']}  |  Target position: {row['target_new_position']} (0-based)")
    print(f"Citation order: {list(row['citation_order'])}")
    print()
    print("── LLM Response ──")
    print(row["llm_response"])
    print()


Query ID: 112260
Query:    wireless wifi security camera system
Num docs: 10  |  Target position: 6 (0-based)
Citation order: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]

── LLM Response ──
If you're in the market for a wireless WiFi security camera system, here are several top-rated options to consider:

1. **Hiseeu Expandable 8CH 2K Wireless Security Camera System**: This system features four 1296P night vision cameras and a 1TB hard drive, providing an easy setup with remote access via the EseeCloud app. The H.265+ video compression ensures high-quality streaming while saving bandwidth. Additionally, it includes advanced AI human detection to minimize false alarms and allows you to monitor your property anytime, anywhere without monthly fees [1].

2. **Hiseeu 2K Wireless Security Camera System with 3TB Hard Drive**: Ideal for comprehensive monitoring, this system includes eight 3MP cameras with ex